[◀ Notebook 03: Expressions](03_expressions.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 05: Compound Statements ▶](05_compound_statements.ipynb)

# Notebook 04: Simple Statements

A simple statement is a statement that is contained within a single logical line. This notebook details assignment mechanics, starred unpacking, augmented assignment, and the namespace deletion mechanics of `del`.

Official Reference: [Python Language Reference - Simple Statements](https://docs.python.org/3/reference/simple_stmts.html)

## 1. Assignment Mechanics & Target Lists

An assignment statement binds (or rebinds) a name to an object. In Python, assignment is **not** an expression (unlike in C/C++), meaning `x = y = 0` is allowed as a special statement syntax, but you cannot write `if (x = 5):`.

### Evaluation Order
During assignment, Python first evaluates the entire Right-Hand Side (RHS) expression list from left to right, then binds the resulting object(s) to the Left-Hand Side (LHS) targets.

### Starred Unpacking (PEP 3132)
Python supports sequence unpacking. Using a star `*` prefix allows unpacking variable-length sequences. Crucially:
*   Only one starred target is allowed on the LHS.
*   The starred variable always captures a list of all remaining elements.

#### Starred Unpacking Placement Rules
The single starred expression can be placed anywhere in the LHS target list:
-   **Beginning:** `*rest, last = seq` (captures everything except the last element)
-   **Middle:** `first, *middle, last = seq` (captures everything except the first and last elements)
-   **End:** `first, second, *rest = seq` (captures everything after the first two elements)
-   **Execution Check:** Under the hood, Python checks the sequence length: it must be at least `len(LHS) - 1`. If the sequence is too short, Python raises a `ValueError`. It then binds the fixed names and packages the remaining items as a list bound to the starred name.

In [ ]:
# Starred Unpacking Demo
sequence = [1, 2, 3, 4, 5]
first, *middle, last = sequence

print("Starred Unpacking in Middle:")
print("first:", first)   # 1
print("middle:", middle) # [2, 3, 4] (a list!)
print("last:", last)     # 5

# Starred Unpacking at Beginning
*rest, last_item = sequence
print("\nStarred Unpacking at Beginning:")
print("rest:", rest)
print("last_item:", last_item)

# Starred Unpacking at End
first_item, second_item, *rest_items = sequence
print("\nStarred Unpacking at End:")
print("first_item:", first_item)
print("second_item:", second_item)
print("rest_items:", rest_items)

# Starred Unpacking Single-element remaining capture
a, *b = [10]
print(f"\na: {a}, b: {b}") # b is empty list []

# Unpacking ValueError check (sequence too short)
try:
    x, y, *z = [100]
except ValueError as e:
    print(f"\nSuccess! Caught expected ValueError: {e}")


## 2. Mutability in Augmented Assignments (`+=`)

Augmented assignments like `x += y` are syntactically similar to `x = x + y`, but Python handles them differently under the hood depending on the mutability of `x`:

1.  **Immutable Objects (e.g. `str`, `int`, `tuple`):** Since they cannot change in-place, `x += y` is identical to `x = x + y`. It creates a **new object** and re-binds `x` to it. The ID changes.
2.  **Mutable Objects (e.g. `list`):** If the object implements `__iadd__`, `x += y` modifies the object **in-place** (for lists, this behaves like `x.extend(y)`). The ID remains **identical**.

In [ ]:
# Immutable augmented assignment (String)
s = "hello"
id_before_s = id(s)
s += " world"
print("String ID remained same?", id(s) == id_before_s) # False

# Mutable augmented assignment (List)
lst = [1, 2]
id_before_lst = id(lst)
lst += [3, 4] # Modifies list in-place!
print("List ID remained same?", id(lst) == id_before_lst) # True

## 3. The `del` Statement

The `del` statement deletes the binding of a name from the local or global namespace.

### Under the Hood
`del` does **not** directly delete the object from memory. Instead:
1.  It removes the variable name binding from the namespace.
2.  It decrements the target object's reference count by 1.
3.  If the object's reference count drops to 0, the Python garbage collector reclaims the memory.

### Dynamic Namespace Cleanup
Because names are stored in dictionaries representing namespaces, you can dynamically delete bindings:
-   **`del namespace_dict[name]`:** Directly removes the key from the dictionary. If the variable name does not exist, it raises a `KeyError`.
-   **`namespace_dict.pop(name, None)`:** Safely removes the key if it exists, returning a default (such as `None`) if the key is not found, thereby avoiding exceptions.
-   **CPython Implementation Detail:** In module scope (globals), namespaces are true dictionaries, making dynamic modifications straightforward. However, inside functions, local variables are optimized to be stored in a fast array (using `LOAD_FAST` and `STORE_FAST` opcodes) instead of a dictionary. Consequently, modifying `locals()` dynamically (such as deleting items or using `.pop()`) does not always write back to the actual local fast array in standard interpreters.

In [ ]:
my_var = "Clean me up"
print("my_var in globals before del:", "my_var" in globals())

del my_var
print("my_var in globals after del:", "my_var" in globals())

# Attempting to access my_var now raises a NameError
try:
    print(my_var)
except NameError as e:
    print("Success! Raised expected NameError:", e)

# Dynamic Dictionary namespace deletion using del (raises KeyError if missing)
my_dict = {"a": 1, "b": 2}
print("\nOriginal dict:", my_dict)
del my_dict["a"]
print("After del my_dict['a']:", my_dict)

try:
    del my_dict["missing_key"]
except KeyError as e:
    print(f"Success! Caught expected KeyError on del: {e}")

# Safe removal using pop(key, default)
val = my_dict.pop("missing_key", None)
print(f"Safe pop() returned: {val}")


---

## Coding Challenges

Complete the challenges below to master simple statements and unpacking.

In [ ]:
# ==========================================
# Challenge 1: Variable-length Sequence Parser
# ==========================================
# Instructions: Write a function `parse_sensor_data(data)` that takes a list or tuple of data.
# The first item is always a string `sensor_id`.
# The last item is always a string `status`.
# Any values in between represent float-based numeric readings.
# 
# Return a tuple: `(sensor_id, readings_list, status)` where `readings_list` contains the 
# numeric readings as a list. If there are no readings in between, return an empty list.
# Requirements:
# - You MUST use starred unpacking (`*`).

def parse_sensor_data(data: list) -> tuple:
    # TODO: Implement using starred unpacking.
    pass

In [ ]:
# Test assertions for Challenge 1
try:
    assert parse_sensor_data(["SN-101", 22.4, 22.5, 22.3, "OK"]) == ("SN-101", [22.4, 22.5, 22.3], "OK"), "Failed basic parse"
    assert parse_sensor_data(["SN-102", "ERROR"]) == ("SN-102", [], "ERROR"), "Failed empty readings check"
    assert parse_sensor_data(["SN-103", 10.0, "OK"]) == ("SN-103", [10.0], "OK"), "Failed single reading check"
    print("🎉 Challenge 1 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 1 Failed: {e}")

In [ ]:
# ==========================================
# Challenge 2: Memory Address Tracking with Augmented Assignment
# ==========================================
# Instructions: Write a function `track_augmented_assignment_ids()` that performs
# augmented addition (`+=`) on a tuple and a list, and returns a tuple of two booleans:
# `(tuple_id_remained_same, list_id_remained_same)`.
#
# Perform:
# 1. Create a tuple `t = (1, 2)`. Record if ID remains identical after `t += (3, 4)`.
# 2. Create a list `l = [1, 2]`. Record if ID remains identical after `l += [3, 4]`.

def track_augmented_assignment_ids() -> tuple:
    # TODO: Implement and return (tuple_id_remained_same, list_id_remained_same).
    pass

In [ ]:
# Test assertions for Challenge 2
try:
    t_same, l_same = track_augmented_assignment_ids()
    assert t_same == False, "Tuple += should create a new object (immutable)"
    assert l_same == True, "List += should mutate in-place (mutable)"
    print("🎉 Challenge 2 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 2 Failed: {e}")

In [ ]:
# ==========================================
# Challenge 3: Safe Namespace Cleaning
# ==========================================
# Instructions: Write a function `clear_namespace(var_names, namespace_dict)`
# that takes a list of variable name strings and a dictionary representing a namespace 
# (like `globals()`). It must delete these variables from the dictionary.
# If any of the variables do not exist in the dictionary, ignore them without raising an error.
# Return the modified dictionary.

def clear_namespace(var_names: list, namespace_dict: dict) -> dict:
    # TODO: Implement namespace cleaning.
    pass

In [ ]:
# Test assertions for Challenge 3
try:
    ns = {"a": 1, "b": 2, "c": 3, "__builtins__": None}
    cleaned = clear_namespace(["a", "c", "z"], ns)
    
    assert "a" not in cleaned, "Variable 'a' was not deleted"
    assert "c" not in cleaned, "Variable 'c' was not deleted"
    assert "b" in cleaned, "Variable 'b' was incorrectly deleted"
    print("🎉 Challenge 3 Passed!")
except AssertionError as e:
    print(f"❌ Challenge 3 Failed: {e}")

[◀ Notebook 03: Expressions](03_expressions.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 05: Compound Statements ▶](05_compound_statements.ipynb)